In [0]:
%sql
create schema if not exists weather.bronze;

In [0]:
# Define your volume path and table names
volume_path = "/Volumes/weather/landing/files/forecast_weather"
target_table= "weather.bronze.forecast_weather"

In [0]:
from pyspark.sql import functions as f
from pyspark.sql.functions import current_timestamp, input_file_name

checkpoint_location = "/Volumes/weather/landing/files/forecast_weather/_checkpoints/"

bronze_df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("cloudFiles.schemaLocation", checkpoint_location)
    .load(volume_path)
)
# 2. Explicitly select the raw fields and inject the hidden metadata columns
bronze_with_metadata_df = bronze_df.select(
    "*",  # Selects all columns inferred from your JSON
    bronze_df["_metadata.file_name"].alias("source_file_name"),
    bronze_df["_metadata.file_modification_time"].alias("source_file_timestamp"),
    bronze_df["_metadata.file_path"].alias("source_file_path"),
).withColumn("ingestion_timestamp", f.current_timestamp())

# Write the stream to the Bronze Delta table
query = (
    bronze_with_metadata_df.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_location)
    .trigger(availableNow=True)
    .toTable(target_table)
)

In [0]:
%sql
select * from weather.bronze.forecast_weather